# AI4SCIENCE Hippocampus Demo

This is a demo of the ai4science(hippocampus) API through the Python client. The following features are demonstrated: 
- Dataset provisioning
- Running OpenML, Hugging Face jobs
- Explicit resource definition
- Working with BYO artifacts
- Automatic tier routing across compute clusters.

## Coming soon

Not yet available through the API -- in progress or planned:

- **Agentic optimization** -- `/agent/optimize` exists server-side (submits
  an agent-driven training/optimization loop to a GPU node), but isn't
  wired into the client yet.
- **Agentic evaluation** -- a dedicated evaluation loop, not yet built.
- **Ray-based multi-node distributed jobs** -- running a job across more
  than one node with Ray for coordination. Not yet available; single-node
  jobs only for now.

Set `AI4SCIENCE_USER` and `AI4SCIENCE_TOKEN` below before running.

In [1]:
import os

BASE_URL = "https://ai4science.dev.sdp.surf.nl"
USER = os.environ.get("AI4SCIENCE_USER", "")
TOKEN = os.environ.get("AI4SCIENCE_TOKEN", "") # authoritzation is a work in progress, token is used as a placeholder

assert USER, "Set AI4SCIENCE_USER (your Snellius username) before running this notebook."
assert TOKEN, "Set AI4SCIENCE_TOKEN (a fresh SLURM JWT, e.g. via scontrol token) before running this notebook."

print(f"Ready. base_url={BASE_URL}, user={USER}")

Ready. base_url=https://ai4science.dev.sdp.surf.nl, user=juliusa


## Client setup

One client instance, reused across every example below.

In [2]:
from ai4science_client import Ai4ScienceClient

client = Ai4ScienceClient(base_url=BASE_URL, user=USER, token=TOKEN)
print("Client ready.")

Client ready.


## FEATURE: Data provisioning

Stage a public dataset onto Snellius via the raw API (`POST /datasets/add`),
then poll `POST /datasets/status` until it's ready. 

Ideally there is an approval/guardrail step between request and staging.

This has been skipped here for demo.

Also perhaps we could have a silent auto garbage collector on unused datasets.

In [11]:
import time

import requests

DATASET_ID, PLATFORM = "GPT-NL/GPT-NL_Public_Corpus", "Hugging Face"
clean_name = DATASET_ID.split("/")[-1]

### PROVISION REQUEST ######
resp = requests.post(f"{BASE_URL}/datasets/add", json={
    "dataset_id": DATASET_ID, "platform": PLATFORM, "user": USER, "token": TOKEN,
})
resp.raise_for_status()
submitted = resp.json()
print("Provisioning job submitted:\n", submitted)

##### STATUS REQUEST #######
state = "PENDING"
while state not in ("READY", "FAILED"):
    time.sleep(10)
    registry = requests.post(f"{BASE_URL}/datasets/status", json={"datasets": [clean_name]}).json()
    state = registry.get(clean_name, {}).get("status", "unknown").upper()
    print("\n\nFinal provisioning status:", state)

Provisioning job submitted:
 {'job_id': 25781449, 'status': 'SUBMITTED', 'output_file': '/projects/2/managed_datasets/containers/ai4science/jobs/25781449.log', 'error_file': '/projects/2/managed_datasets/containers/ai4science/jobs/25781449.log', 'user': 'wodsmgr', 's3_key': None}


Final provisioning status: READY


## FEATURE: OpenML job via the `@job` decorator

A real OpenML workflow -- fetch a task, train a model, report the score --
running on Snellius. 

`@job(...)` turns a plain Python function into a
remote job with near-zero snellius env config at the call site.

In [6]:
from ai4science_client import job


@job(base_url=BASE_URL, user=USER, token=TOKEN, dependencies=["openml", "scikit-learn"], stream=True) # only line to run job everything else is user code
def run_openml_task(task_id: int) -> dict:
    import time
    import openml
    from sklearn.ensemble import RandomForestClassifier

    def get_task_with_retries(task_id, attempts=3, delay=10):
        for attempt in range(1, attempts + 1):
            try:
                return openml.tasks.get_task(task_id)
            except Exception as e:
                print(f"openml.org fetch failed (attempt {attempt}/{attempts}): {e}")
                if attempt == attempts:
                    raise
                time.sleep(delay)

    task = get_task_with_retries(task_id)
    X, y = task.get_X_and_y()
    train_idx, test_idx = task.get_train_test_split_indices()

    clf = RandomForestClassifier(n_estimators=100)
    clf.fit(X[train_idx], y[train_idx])
    accuracy = clf.score(X[test_idx], y[test_idx])

    return {"task_id": task_id, "accuracy": accuracy}


result = run_openml_task(31)
print(result)

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): openml scikit-learn ===
INFO   : A system administrator may need to enable user namespaces, install
INFO   :   apptainer-suid, or compile with ./mconfig --with-suid
ERROR  : Failed to create user namespace: maximum number of user namespaces exceeded, check /proc/sys/user/max_user_namespaces


NOTICE: Apptainer container isolation is currently unavailable on this node
(unprivileged user namespaces are disabled -- a known, temporary Snellius
restriction). This job is falling back to an isolated Python virtual
environment instead of a container. Your job will still run with a clean,
job-specific set of installed dependencies, matching the container's base
stack, and nothing installed here affects any other job, user, or the base
environment. This fallback exists so ephemeral jobs keep working while
container support is restored.

=== [2b/3] Creating isolated venv for this job ===
Using CPyth

## FEATURE: Hugging Face job with explicit resources + a local artifact

Run a Hugging Face model with explicit resource requirements
(`SlurmResourceConfig`).

And pass in a local file as an artifact -- uploaded
automatically, available inside the job as a normal function argument, no
extra code needed to fetch it.

In [5]:
with open("sample_texts.txt", "w") as f:
    f.write("Snellius makes large-scale AI research so much easier.\n")
    f.write("I really dislike waiting for slow pip installs.\n")
    f.write("This dataset looks promising for our next experiment.\n")

In [4]:
from ai4science_client.schemas import SlurmResourceConfig

resources = SlurmResourceConfig(
    partition="gpu_h100",
    cpus_per_task=8,
    memory_mb=32000,
    time_limit_minutes=20,
    tres_per_node="gres:gpu:1",
)


def classify_with_local_texts(data_path: str) -> dict:
    import torch
    from transformers import pipeline

    with open(data_path) as f:
        texts = [line.strip() for line in f if line.strip()]

    device = 0 if torch.cuda.is_available() else -1
    classifier = pipeline("sentiment-analysis", device=device)
    results = classifier(texts)

    return {"texts": texts, "results": results}


result = client.run(
    classify_with_local_texts,
    artifacts={"data_path": "./sample_texts.txt"},
    dependencies=["torch", "transformers<4.50"],
    resources=resources,
    stream=True,
)

print(result)

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): torch transformers<4.50 boto3 ===
INFO   : A system administrator may need to enable user namespaces, install
INFO   :   apptainer-suid, or compile with ./mconfig --with-suid
ERROR  : Failed to create user namespace: maximum number of user namespaces exceeded, check /proc/sys/user/max_user_namespaces


NOTICE: Apptainer container isolation is currently unavailable on this node
(unprivileged user namespaces are disabled -- a known, temporary Snellius
restriction). This job is falling back to an isolated Python virtual
environment instead of a container. Your job will still run with a clean,
job-specific set of installed dependencies, matching the container's base
stack, and nothing installed here affects any other job, user, or the base
environment. This fallback exists so ephemeral jobs keep working while
container support is restored.

=== [2b/3] Creating isolated venv for this job ===
U

## 4. Automatic tier routing

Pass `tier="auto"` and the API estimates what a job actually needs (CPU,
memory, GPU) from its script and dependencies, then picks the smallest
compute cluster that can satisfy it -- searching every tier, cluster and partitions available for the best fit. 
    
Here, a GPU-needing job is checked against both `dev-slurm` (local slurm sim with no
GPU capability) and `snellius` (has GPU partitions) -- only Snellius can
satisfy it, so that's where it lands :))

You can also control how much of that decision the API makes for you:

```python
# Fully automatic -- search everything, pick the smallest fit
client.run(fn, tier="auto")

# Pin to a tier -- only search clusters within it
client.run(fn, tier="1")

# Pin to a specific cluster -- skip the tier/cluster search, still
# picks the smallest fitting partition on that one cluster
client.run(fn, tier="1", cluster="snellius")

# Pin cluster + exact partition -- no estimation or search at all,
# runs exactly where you say
client.run(
    fn,
    tier="1",
    cluster="snellius",
    resources=SlurmResourceConfig(partition="gpu_h100"),
)
```

In [3]:
def gpu_matrix_multiply(size: int = 2048) -> dict:
    import time

    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)

    torch.cuda.synchronize() if device == "cuda" else None
    start = time.time()
    c = a @ b
    torch.cuda.synchronize() if device == "cuda" else None
    elapsed = time.time() - start

    return {"device": device, "size": size, "elapsed_seconds": elapsed}


result = client.run(
    gpu_matrix_multiply,
    2048,
    dependencies=["torch"],
    tier="auto",
    stream=True,
)
print(result)

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): torch ===
INFO   : A system administrator may need to enable user namespaces, install
INFO   :   apptainer-suid, or compile with ./mconfig --with-suid
ERROR  : Failed to create user namespace: maximum number of user namespaces exceeded, check /proc/sys/user/max_user_namespaces


NOTICE: Apptainer container isolation is currently unavailable on this node
(unprivileged user namespaces are disabled -- a known, temporary Snellius
restriction). This job is falling back to an isolated Python virtual
environment instead of a container. Your job will still run with a clean,
job-specific set of installed dependencies, matching the container's base
stack, and nothing installed here affects any other job, user, or the base
environment. This fallback exists so ephemeral jobs keep working while
container support is restored.

=== [2b/3] Creating isolated venv for this job ===
Using CPython 3.11.13 int

## FEATURE: EESSI module-based job (raw API)

EESSI jobs use pre-built software modules instead of installing
dependencies at runtime -- a different execution model from
`/ephemeral-job`. 

The client doesn't wrap this one yet, so this is a
plain `POST /eessi-job` request, same shape as the dataset provisioning
call above.

In [10]:
resp = requests.post(
    f"{BASE_URL}/eessi-job",
    json={
        "modules": ["foss/2023a", "SciPy-bundle/2023.07-gfbf-2023a"],
        "python_script": (
            "import numpy as np\n"
            "a = np.random.rand(1000, 1000)\n"
            "print('Matrix sum:', a.sum())\n"
        ),
        "user": USER,
        "token": TOKEN,
    },
)
resp.raise_for_status()
submitted = resp.json()
print("EESSI job submitted:", submitted)

job_id = submitted["job_id"]
seen_len = 0
state = "pending"

while state not in ("completed", "failed"):
    time.sleep(10)

    log_resp = requests.get(f"{BASE_URL}/logs/{job_id}")
    if log_resp.status_code == 200 and len(log_resp.text) > seen_len:
        print(log_resp.text[seen_len:], end="")
        seen_len = len(log_resp.text)

    result_resp = requests.get(f"{BASE_URL}/results/{job_id}")
    if result_resp.status_code == 404:
        continue
    result_resp.raise_for_status()
    result = result_resp.json()
    state = result.get("status", "unknown")

print("\nFinal result:", result)

EESSI job submitted: {'job_id': 25784779, 'status': 'SUBMITTED', 'output_file': '/tmp/25784779.log', 'error_file': '/tmp/25784779.log', 'user': 'juliusa', 's3_key': None}
--- Phase 1: EESSI Init ---
Attempting to initialize EESSI version: 2023.06
EESSI Initialized successfully.
--- Phase 2: Module Load ---
Loading foss/2023a...
Loading SciPy-bundle/2023.07-gfbf-2023a...
--- Phase 3: Python ---
Using Python from: /cvmfs/software.eessi.io/versions/2023.06/software/linux/x86_64/amd/zen4/software/Python/3.11.3-GCCcore-12.3.0/bin/python3
Matrix sum: 500558.9620113597
=== Job Finished with exit code 0 ===

Final result: {'job_id': '25784779', 'status': 'completed', 'exit_code': 0, 'result': None}


## Other features available

Not demoed above, but supported by the client and API today:

- **Async submission** -- `client.submit(...)` returns immediately; poll or
  block on it later with `job.results()` / `job.wait()`.

- **Hugging Face/openml token passthrough** -- `hf_token=` on `run()`/`@job(...)`
  for gated/private HF models and datasets.